# Notebook 05: Feature Engineering Patterns

## Why This Matters

Feature engineering is where most of the performance gain happens in tabular ML - much more than model choice. The difference between a Kaggle competitor finishing top 10% vs top 1% is almost always feature engineering. In production, thoughtful feature engineering also improves model interpretability, reduces overfitting, and makes features more robust to distribution shift. This notebook covers the canonical patterns: numerical, categorical, temporal, text, and geospatial features, with a focus on trade-offs and leakage traps.

In [ ]:
%pip install -q scikit-learn pandas numpy matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import (StandardScaler, PowerTransformer, KBinsDiscretizer, OneHotEncoder)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
import warnings; warnings.filterwarnings("ignore")
np.random.seed(42)
print("Ready.")

## 1. Numerical Feature Transformations

**Why transform?** Many models assume roughly Gaussian features. Log/power transforms reduce the impact of extreme outliers.

**Binning strategies**:
- **Equal-width bins**: same range per bin. Bad for skewed distributions.
- **Equal-frequency (quantile) bins**: same number of samples per bin. Robust to skew. Use by default.

**Log transform**: `log(x+1)` for non-negative skewed features (income, prices, counts).

**Power transform (Yeo-Johnson)**: finds optimal power lambda to make distribution Gaussian. Handles negative values.

In [ ]:
np.random.seed(42)
income = np.exp(np.random.normal(10, 1, 2000))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
transforms = {
    "Original": income,
    "Log(x+1)": np.log1p(income),
    "Sqrt": np.sqrt(income),
    "Equal-width bins (10)": KBinsDiscretizer(n_bins=10, encode="ordinal", strategy="uniform").fit_transform(income.reshape(-1,1)).ravel(),
    "Equal-freq bins (10)": KBinsDiscretizer(n_bins=10, encode="ordinal", strategy="quantile").fit_transform(income.reshape(-1,1)).ravel(),
    "Power (Yeo-Johnson)": PowerTransformer(method="yeo-johnson").fit_transform(income.reshape(-1,1)).ravel(),
}
for ax, (name, data) in zip(axes.ravel(), transforms.items()):
    ax.hist(data, bins=50, edgecolor="black", linewidth=0.5)
    ax.set_title(f"{name} (skew={pd.Series(data).skew():.2f})")
plt.suptitle("Numerical Feature Transformations: Reducing Skew")
plt.tight_layout(); plt.savefig("/tmp/num_transforms.png", dpi=80); plt.show()

## 2. Categorical Feature Encoding

**One-hot encoding**: binary column per category. Good for low-cardinality (<50 unique values). Bad for high cardinality.

**Ordinal encoding**: integer per category. ONLY appropriate when categories have natural order (Low/Medium/High). Dangerous on nominal categories.

**Target encoding**: replace category with mean(target). Very powerful for high-cardinality.
**Critical leakage trap**: computing target mean on full dataset before splitting leaks labels into features.
**Safe implementation**: use k-fold cross-validated mean (estimate within each training fold only).

In [ ]:
np.random.seed(42)
n = 5000
countries = ["US","UK","DE","FR","CN","BR","IN","RU"] * (n//8)
fraud_rates = {"US":0.03,"UK":0.04,"DE":0.02,"FR":0.03,"CN":0.08,"BR":0.07,"IN":0.05,"RU":0.09}
df = pd.DataFrame({"country": countries[:n]})
df["fraud"] = df.country.map(lambda c: np.random.binomial(1, fraud_rates[c]))
X_tr, X_te, y_tr, y_te = train_test_split(df, df.fraud, test_size=0.2, stratify=df.fraud, random_state=42)

# LEAKY: compute on all training data at once
leaky_means = X_tr.groupby("country")["fraud"].mean()
X_tr["enc_leaky"] = X_tr.country.map(leaky_means)
X_te["enc_leaky"] = X_te.country.map(leaky_means).fillna(y_tr.mean())

# SAFE: cross-validated target encoding
from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
safe_enc = pd.Series(index=X_tr.index, dtype=float)
global_mean = y_tr.mean()
for trn_idx, val_idx in kf.split(X_tr):
    trn_df = X_tr.iloc[trn_idx]; val_df = X_tr.iloc[val_idx]; trn_y = y_tr.iloc[trn_idx]
    cat_means = trn_df.assign(y=trn_y.values).groupby("country")["y"].agg(["mean","count"])
    cat_means["smoothed"] = (cat_means["mean"]*cat_means["count"]+global_mean*10)/(cat_means["count"]+10)
    safe_enc.iloc[val_idx] = val_df.country.map(cat_means["smoothed"]).fillna(global_mean).values
X_tr["enc_safe"] = safe_enc

train_means_safe = X_tr.assign(y=y_tr.values).groupby("country").apply(
    lambda g: (g["y"].mean()*len(g)+global_mean*10)/(len(g)+10))
X_te["enc_safe"] = X_te.country.map(train_means_safe).fillna(global_mean)

true_rates = pd.Series(fraud_rates)
leaky_corr = X_te.groupby("country")["enc_leaky"].mean().corr(true_rates)
safe_corr = X_te.groupby("country")["enc_safe"].mean().corr(true_rates)
print("=== Target Encoding Quality ===")
print(f"Leaky encoding correlation with true fraud rate: {leaky_corr:.4f}")
print(f"Safe encoding correlation with true fraud rate:  {safe_corr:.4f}")
print("The key difference: safe encoding uses only training fold data.")

## 3. Date/Time Feature Engineering

Temporal features are among the highest-value features in many models. Key patterns:

- **Cyclical features**: hour, day-of-week, month - encode as sin/cos to preserve cyclical distance
- **Recency features**: time since last event - captures freshness, urgency
- **Calendar features**: is_weekend, is_holiday, is_end_of_month

**Why sin/cos encoding**: hour 23 and hour 0 are adjacent, but naive encoding (23, 0) shows them as far apart. Cyclical encoding preserves this adjacency.

In [ ]:
dates = pd.date_range("2023-01-01", periods=365*24, freq="h")
df_ts = pd.DataFrame({"timestamp": dates})
df_ts["hour"] = df_ts.timestamp.dt.hour

# Cyclical encoding
df_ts["hour_sin"] = np.sin(2 * np.pi * df_ts.hour / 24)
df_ts["hour_cos"] = np.cos(2 * np.pi * df_ts.hour / 24)

# Distance demonstration
hour_23 = np.array([np.sin(2*np.pi*23/24), np.cos(2*np.pi*23/24)])
hour_0  = np.array([np.sin(2*np.pi*0/24),  np.cos(2*np.pi*0/24)])
hour_12 = np.array([np.sin(2*np.pi*12/24), np.cos(2*np.pi*12/24)])
print("=== Cyclical Encoding: Hour Feature ===")
print(f"Naive: distance(23,0) = 23, distance(23,12) = 11")
print(f"Cyclical: distance(23,0) = {np.linalg.norm(hour_23-hour_0):.3f}, distance(23,12) = {np.linalg.norm(hour_23-hour_12):.3f}")
print("Cyclical correctly represents 23:00 and 00:00 as adjacent.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
hours = np.arange(24)
axes[0].scatter(np.sin(2*np.pi*hours/24), np.cos(2*np.pi*hours/24), c=hours, cmap="hsv")
for h in [0, 6, 12, 18]: axes[0].annotate(str(h), (np.sin(2*np.pi*h/24), np.cos(2*np.pi*h/24)))
axes[0].set_title("Cyclical Hour: (sin, cos)")
axes[1].plot(hours, np.sin(2*np.pi*hours/24), label="sin"); axes[1].plot(hours, np.cos(2*np.pi*hours/24), label="cos")
axes[1].set_title("Sin/Cos Components"); axes[1].legend()
plt.tight_layout(); plt.savefig("/tmp/cyclical.png", dpi=80); plt.show()

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Log/Power transform | Reduces skew; Yeo-Johnson is automatic and handles negatives |
| Equal-frequency bins | Better than equal-width for skewed distributions |
| One-hot encoding | Low-cardinality categoricals |
| Target encoding | High-cardinality categoricals; use k-fold CV to prevent leakage |
| Cyclical features | Sin/cos encoding for cyclic features (hour, month, day-of-week) |
| Interaction features | Explicit cross-products for linear models; trees discover automatically |

### Common Pitfalls
- Applying target encoding without cross-validation (direct leakage)
- Using ordinal encoding on nominal categories (encodes false order)
- Not encoding cyclical features as sin/cos
- Computing bin edges on the full dataset before splitting (subtle leakage)
